In [ ]:
!pip install transformers
!pip install pyannote.audio

In [1]:
import torch
#from .autonotebook import tqdm as notebook_tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline
from whisperx_numpy2_compatibility import load_align_model, align
from whisperx_numpy2_compatibility.diarize import assign_word_speakers
from pyannote.core import Segment


import textwrap
import os
import logging

#os.environ['CURL_CA_BUNDLE'] = ''

c:\Projects\speech_recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []


In [2]:
def find_intersections(speakers, texts):
    intersections = []

    for text in texts:
        text_start, text_end = text['start'], text['end']-0.1
        for turn, _, speaker in speakers.itertracks(yield_label=True):
            speaker_start, speaker_end = turn.start, turn.end
            
            # Find the overlap between the speaker's interval and the text's interval
            start = max(text_start, speaker_start)
            end = min(text_end, speaker_end)
            
            if start < end:  # There is an intersection
                if intersections and intersections[-1]['speaker'] == speaker:
                    intersections[-1]['end'] = end
                    intersections[-1]['text'] += ' ' + text['text']
                else:
                    intersections.append({
                        'start': start,
                        'end': end,
                        'speaker': speaker,
                        'text': text['text']
                    })
    return intersections


In [3]:
LOCAL_MODEL = False

In [4]:
def merge_speech_segments(segments):
    merged_segments = []
    for segment in segments:
        if merged_segments and segment["speaker"] == merged_segments[-1]["speaker"]:
            # Extend the end time and append text for the same speaker
            merged_segments[-1]["end"] = segment["end"]
            merged_segments[-1]["text"] += " " + segment["text"]
        else:
            # Add a new segment if the speaker changes
            merged_segments.append(segment)
    return merged_segments


In [5]:
def save_speech_to_file_with_indent(segments, filename):
    with open(filename, "w", encoding="utf-8") as file:
        for segment in segments:
            # Format the speaker tag
            speaker_tag = f"{segment['speaker'].upper()}:\n"
            
            # Wrap the text to 128 characters and indent each line
            wrapped_text = textwrap.fill(segment["text"], width=128, subsequent_indent="    ")
            
            # Write the formatted text to the file
            file.write(speaker_tag)
            file.write(wrapped_text)
            file.write("\n\n")  # Add a blank line between speakers


In [6]:
HF_TOKEN="XXXXXX"

if LOCAL_MODEL:
    DIARIZATION_MODEL="/Projects/AI/models/speaker-diarization-3.1/config.yaml"
    align_model="/Projects/AI/models/wav2vec2-large-xlsr-53-russian/"
else:
    DIARIZATION_MODEL="pyannote/speaker-diarization-3.1"
    align_model='jonatasgrosman/wav2vec2-large-xlsr-53-russian'

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(device)

cpu


In [8]:
#Initializing up wisper pipeline
whisper_model_id="openai/whisper-large-v3"
#whisper_model_id="openai/whisper-medium"
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    whisper_model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
whisper_model.to(device)
whisper_processor = AutoProcessor.from_pretrained(whisper_model_id)
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    chunk_length_s=30,  # Process audio in 30-second chunks
    stride_length_s=10,  # Optional overlap between chunks    
    torch_dtype=torch_dtype,
    device=device,
)


Device set to use cpu


In [9]:
labels = whisper_processor.tokenizer.get_vocab()
align_dictionary = {char.lower(): code for char,code in whisper_processor.tokenizer.get_vocab().items()}

In [10]:
diarization_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))
#model = whisper.load_model(WHISPER_MODEL, download_root='./models', device=DEVICE)

C:\Program Files\Python312\Lib\inspect.py:1007: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):


In [11]:
file_name='audio/2407151757656693.1.0.0.mp3'
script = whisper_pipe(file_name, return_timestamps='word', generate_kwargs={"language": "russian"})
#script = whisper_pipe(file_name, return_timestamps=True, generate_kwargs={"language": "russian"})
print(script['chunks'])

c:\Projects\speech_recognition\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed language=russian, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of language=russian.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
WhisperModel is using WhisperSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `layer_head_mask` not None. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be re

KeyboardInterrupt: 

In [66]:
script

{'text': ' ЗВОНОК ТЕЛЕФОНА Здравствуйте. ООО «Интерлизинг» уведомляет вас о просроченной задолженности в размере в 100 000 рублей по лизинговому договору. Заключенному Слауадерстрой. Предмет лизинга – ревковой автомобиль «Ладокс Рай». Упросим вас незамедлительно произвести оплату. Если вы уже оплатили просроченную задолженность, скажите «Да». Нет. Если у вас остались вопросы или вам необходима консультация специалиста, скажите «Да» или «Нет». Да. С уважением к вам и к вашему бизнесу. Интерлизинг.',
 'chunks': [{'timestamp': (0.0, 22.0),
   'text': ' ЗВОНОК ТЕЛЕФОНА Здравствуйте. ООО «Интерлизинг» уведомляет вас о просроченной задолженности в размере в 100 000 рублей по лизинговому договору. Заключенному Слауадерстрой. Предмет лизинга – ревковой автомобиль «Ладокс Рай».'},
  {'timestamp': (22.56, 29.24),
   'text': ' Упросим вас незамедлительно произвести оплату. Если вы уже оплатили просроченную задолженность, скажите «Да».'},
  {'timestamp': (29.24, 29.62), 'text': ' Нет.'},
  {'times

In [37]:
diarized = diarization_pipeline(file_name, min_speakers=1, max_speakers=3)

c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_samp

In [79]:
print(diarized)

[ 00:00:02.916 -->  00:00:04.030] A SPEAKER_02
[ 00:00:11.843 -->  00:00:25.022] B SPEAKER_00
[ 00:00:25.360 -->  00:00:28.262] C SPEAKER_00
[ 00:00:29.292 -->  00:00:29.612] D SPEAKER_02
[ 00:00:32.110 -->  00:00:37.560] E SPEAKER_00
[ 00:00:38.775 -->  00:00:39.096] F SPEAKER_01
[ 00:00:40.615 -->  00:00:43.399] G SPEAKER_00


In [106]:
for turn, _, speaker_label in diarized.itertracks(yield_label=True):
    print(f'{turn.start} - {turn.end}: {speaker_label}')


2.91659375 - 4.03034375: SPEAKER_02
11.84346875 - 25.02284375: SPEAKER_00
25.360343750000002 - 28.262843750000002: SPEAKER_00
29.29221875 - 29.612843750000003: SPEAKER_02
32.11034375 - 37.56096875: SPEAKER_00
38.775968750000004 - 39.096593750000004: SPEAKER_01
40.61534375 - 43.399718750000005: SPEAKER_00


In [81]:
script['chunks']

[{'text': ' Здравствуйте!', 'timestamp': (11.32, 12.56)},
 {'text': ' ООО', 'timestamp': (12.56, 12.9)},
 {'text': ' «Интерлидинг»', 'timestamp': (12.9, 13.5)},
 {'text': ' уведомляет', 'timestamp': (13.5, 14.04)},
 {'text': ' вас', 'timestamp': (14.04, 14.2)},
 {'text': ' о', 'timestamp': (14.2, 14.32)},
 {'text': ' просроченной', 'timestamp': (14.32, 14.9)},
 {'text': ' задолженности', 'timestamp': (14.9, 15.54)},
 {'text': ' в', 'timestamp': (15.54, 15.66)},
 {'text': ' размере', 'timestamp': (15.66, 16.04)},
 {'text': ' в', 'timestamp': (16.04, 16.06)},
 {'text': ' 100', 'timestamp': (16.06, 16.18)},
 {'text': ' 000', 'timestamp': (16.18, 16.42)},
 {'text': ' рублей', 'timestamp': (16.42, 16.76)},
 {'text': ' по', 'timestamp': (16.76, 16.92)},
 {'text': ' лизинговому', 'timestamp': (16.92, 17.46)},
 {'text': ' договору,', 'timestamp': (17.46, 18.06)},
 {'text': ' заключенному', 'timestamp': (18.06, 18.78)},
 {'text': ' с', 'timestamp': (18.78, 18.88)},
 {'text': ' ООО', 'timestamp'

In [107]:
# Combine results
speaker_transcription = []
for chunk in script['chunks']:
    start_time, end_time = chunk["timestamp"][0], chunk["timestamp"][1]
    speaker = "Unknown"
    for turn, _, speaker_label in diarized.itertracks(yield_label=True):
        if turn.start <= start_time <= turn.end or turn.start <= end_time <= turn.end :
            speaker = speaker_label
            break
    speaker_transcription.append({
        "start": start_time,
        "end": end_time,
        "speaker": speaker,
        "text": chunk["text"]
    })

In [104]:
list(diarized.itertracks())

[(<Segment(2.91659, 4.03034)>, 'A'),
 (<Segment(11.8435, 25.0228)>, 'B'),
 (<Segment(25.3603, 28.2628)>, 'C'),
 (<Segment(29.2922, 29.6128)>, 'D'),
 (<Segment(32.1103, 37.561)>, 'E'),
 (<Segment(38.776, 39.0966)>, 'F'),
 (<Segment(40.6153, 43.3997)>, 'G')]

In [108]:
speaker_transcription

[{'start': 11.32,
  'end': 12.56,
  'speaker': 'SPEAKER_00',
  'text': ' Здравствуйте!'},
 {'start': 12.56, 'end': 12.9, 'speaker': 'SPEAKER_00', 'text': ' ООО'},
 {'start': 12.9,
  'end': 13.5,
  'speaker': 'SPEAKER_00',
  'text': ' «Интерлидинг»'},
 {'start': 13.5, 'end': 14.04, 'speaker': 'SPEAKER_00', 'text': ' уведомляет'},
 {'start': 14.04, 'end': 14.2, 'speaker': 'SPEAKER_00', 'text': ' вас'},
 {'start': 14.2, 'end': 14.32, 'speaker': 'SPEAKER_00', 'text': ' о'},
 {'start': 14.32,
  'end': 14.9,
  'speaker': 'SPEAKER_00',
  'text': ' просроченной'},
 {'start': 14.9,
  'end': 15.54,
  'speaker': 'SPEAKER_00',
  'text': ' задолженности'},
 {'start': 15.54, 'end': 15.66, 'speaker': 'SPEAKER_00', 'text': ' в'},
 {'start': 15.66, 'end': 16.04, 'speaker': 'SPEAKER_00', 'text': ' размере'},
 {'start': 16.04, 'end': 16.06, 'speaker': 'SPEAKER_00', 'text': ' в'},
 {'start': 16.06, 'end': 16.18, 'speaker': 'SPEAKER_00', 'text': ' 100'},
 {'start': 16.18, 'end': 16.42, 'speaker': 'SPEAKER_

In [18]:
def transcript(file_name):
    logging.info('started')
    script = whisper_pipe(file_name, return_timestamps='word', generate_kwargs={"language": "russian"})
    with open('script_2.txt', "w", encoding="utf-8") as f:
        f.write(script["text"])    
    logging.info('loaded')
    diarized = diarization_pipeline(file_name, min_speakers=1, max_speakers=9)
    logging.info(diarized)
    # Combine results
    speaker_transcription = []
    for chunk in script['chunks']:
        start_time, end_time = chunk["timestamp"][0], chunk["timestamp"][1]
        speaker = "Unknown"
        for turn, _, speaker_label in diarized.itertracks(yield_label=True):
            if turn.start <= start_time <= turn.end or turn.start <= end_time <= turn.end :
                speaker = speaker_label
                break
        speaker_transcription.append({
            "start": start_time,
            "end": end_time,
            "speaker": speaker,
            "text": chunk["text"]
        })
    transcribed = []
    for segment in speaker_transcription:
        transcribed.append(
            {
                "start": segment["start"],
                "end": segment["end"],
                "text": segment["text"],
                "speaker": segment["speaker"] if 'speaker' in segment else "ND"
            }
        )

    merged = merge_speech_segments(transcribed)

    out_file, _ = os.path.splitext(file_name)
    out_file = f"{out_file}_transcript_2.txt"
    save_speech_to_file_with_indent(merged, out_file)

In [16]:
audios=["audio/2407151757656693.1.0.0.mp3"]#, "./audio/audio1415011527.m4a", "./audio/audio1499365096.m4a"]

In [19]:
for audio in audios:
        transcript(audio)

INFO:root:started
c:\Projects\speech_recognition\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
INFO:root:loaded
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown